# Chapter 4 Tutorial: Mathematical Framework of Statistics, Part II

This notebook turns the core ideas of Chapter 4 into executable examples.

The chapter is about what happens when random variables are **combined**.

Main ideas:

1. Linear combinations of random variables
2. Propagation of mean and variance
3. Variance and standard error of a sample mean
4. Central limit theorem
5. Combining estimates from several samples
6. Weighted averages
7. Pooled variance
8. Derived measurements
9. Bias in non-linear transformations
10. Law of propagation of errors
11. Parameter-estimation viewpoint

The examples are written in a measurement/benchmarking mindset, with physics and chemistry examples from the chapter.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statistics

rng = np.random.default_rng(42)
pd.set_option("display.precision", 5)

## 1. Combining random variables

A measured quantity is often computed from other measured quantities.

Example from the chapter:

\[
d = y - x
\]

where:

- \(x\): measured start time
- \(y\): measured stop time
- \(d\): measured duration

Both \(x\) and \(y\) have uncertainty, so \(d\) has uncertainty too.

Another example:

\[
\bar{x}=\frac{x_1+x_2+\cdots+x_N}{N}
\]

An average is also a combination of random variables.

## 2. Scaling a random variable

If:

\[
Y = kX
\]

then:

\[
E(Y)=kE(X)
\]

and:

\[
V(Y)=k^2V(X)
\]

So multiplying a measurement by 10 multiplies its standard deviation by 10, but its variance by 100.

Example: convert centimeters to millimeters.

In [ ]:
# Simulate measured lengths in centimeters
x_cm = rng.normal(loc=12.0, scale=0.2, size=100_000)

# Convert to millimeters
y_mm = 10 * x_cm

summary = pd.DataFrame({
    "quantity": ["x in cm", "y = 10x in mm"],
    "mean": [x_cm.mean(), y_mm.mean()],
    "variance": [x_cm.var(ddof=0), y_mm.var(ddof=0)],
    "standard_deviation": [x_cm.std(ddof=0), y_mm.std(ddof=0)],
})

summary

The standard deviation scales by 10. The variance scales by \(10^2=100\).

## 3. Linear combinations: expected values

For:

\[
L = aX + bY
\]

the expected value is:

\[
E(L)=aE(X)+bE(Y)
\]

This does **not** require independence.

Example: a measured duration

\[
D = Y - X
\]

If the true start time is 10 seconds and true stop time is 25 seconds, the expected duration is:

\[
25 - 10 = 15
\]

In [ ]:
n = 100_000

true_start = 10.0
true_stop = 25.0

# independent timing errors
start = true_start + rng.normal(0, 0.1, size=n)
stop = true_stop + rng.normal(0, 0.1, size=n)
duration = stop - start

pd.DataFrame({
    "quantity": ["start", "stop", "duration = stop - start"],
    "sample_mean": [start.mean(), stop.mean(), duration.mean()],
    "sample_sd": [start.std(ddof=1), stop.std(ddof=1), duration.std(ddof=1)]
})

## 4. Linear combinations: variances

If \(X\) and \(Y\) are independent:

\[
V(aX+bY)=a^2V(X)+b^2V(Y)
\]

Important:

\[
V(Y-X)=V(Y)+V(X)
\]

The variance of a difference is still a sum if the errors are independent.

This is often counterintuitive at first.

In [ ]:
var_start = start.var(ddof=0)
var_stop = stop.var(ddof=0)
var_duration = duration.var(ddof=0)

pd.DataFrame({
    "quantity": ["V(start)", "V(stop)", "V(duration)", "V(start)+V(stop)"],
    "value": [var_start, var_stop, var_duration, var_start + var_stop]
})

The duration variance is approximately the sum of the start and stop variances.

The minus sign does not subtract variance, because the coefficient is squared:

\[
(-1)^2 = 1
\]

## 5. Covariance: when errors are not independent

For non-independent variables:

\[
V(X+Y)=V(X)+V(Y)+2\operatorname{Cov}(X,Y)
\]

and:

\[
V(Y-X)=V(Y)+V(X)-2\operatorname{Cov}(X,Y)
\]

### Stopwatch interpretation

If the operator has the same reaction delay at start and stop, that common delay cancels in \(Y-X\).

That means positive covariance can reduce the variance of a difference.

In [ ]:
n = 100_000

# common reaction delay affects both start and stop
common_delay = rng.normal(0, 0.2, size=n)

# small independent jitter at start/stop
start_jitter = rng.normal(0, 0.03, size=n)
stop_jitter = rng.normal(0, 0.03, size=n)

start_corr = true_start + common_delay + start_jitter
stop_corr = true_stop + common_delay + stop_jitter
duration_corr = stop_corr - start_corr

cov_start_stop = np.cov(start_corr, stop_corr, ddof=0)[0, 1]

pd.DataFrame({
    "quantity": [
        "V(start)",
        "V(stop)",
        "Cov(start, stop)",
        "V(duration)",
        "V(start)+V(stop)-2Cov(start,stop)"
    ],
    "value": [
        start_corr.var(ddof=0),
        stop_corr.var(ddof=0),
        cov_start_stop,
        duration_corr.var(ddof=0),
        start_corr.var(ddof=0) + stop_corr.var(ddof=0) - 2 * cov_start_stop
    ]
})

The start and stop times are individually noisy because of the common delay.

But the measured duration is much less noisy because the common delay cancels.

## 6. Variance of a sample mean

If \(X_1,\ldots,X_N\) are independent observations from the same population:

\[
V(\bar{X})=\frac{V(X)}{N}
\]

and:

\[
\sigma_{\bar{X}}=\frac{\sigma}{\sqrt{N}}
\]

This is the **standard error of the mean**.

Averaging reduces random error, but only by \(\sqrt{N}\).

In [ ]:
true_mu = 100.0
true_sigma = 5.0

def simulate_sample_means(sample_size, n_samples=20_000):
    samples = rng.normal(true_mu, true_sigma, size=(n_samples, sample_size))
    return samples.mean(axis=1)

rows = []
for N in [1, 2, 4, 10, 25, 100]:
    means = simulate_sample_means(N)
    rows.append({
        "N": N,
        "observed_sd_of_sample_means": means.std(ddof=1),
        "theoretical_standard_error": true_sigma / math.sqrt(N)
    })

pd.DataFrame(rows)

In [ ]:
for N in [1, 4, 25]:
    means = simulate_sample_means(N)
    plt.figure()
    plt.hist(means, bins=50)
    plt.xlabel("sample mean")
    plt.ylabel("count")
    plt.title(f"Distribution of sample means, N={N}")
    plt.show()

### Measurement interpretation

If one measurement has standard deviation 5 units:

- average of 4 measurements has standard error \(5/\sqrt{4}=2.5\)
- average of 25 measurements has standard error \(5/\sqrt{25}=1\)
- average of 100 measurements has standard error \(5/\sqrt{100}=0.5\)

To reduce random error by 10×, you need 100× more measurements.

## 7. Central limit theorem

The central limit theorem says that averages tend to become approximately normal even when the original variable is not normal.

The chapter uses a uniform-like birthday-day example.

Here we use a discrete uniform distribution:

\[
X \in \{1,2,\ldots,30\}
\]

A single observation is flat, not normal.

But averages of several observations become bell-shaped.

In [ ]:
def sample_uniform_means(N, n_samples=50_000):
    samples = rng.integers(1, 31, size=(n_samples, N))
    return samples.mean(axis=1)

for N in [1, 2, 5, 10, 30]:
    means = sample_uniform_means(N)
    plt.figure()
    plt.hist(means, bins=50)
    plt.xlabel("average")
    plt.ylabel("count")
    plt.title(f"Averages of N={N} uniform birthday-day values")
    plt.show()

### Interpretation

For \(N=1\), all values from 1 to 30 are roughly equally likely.

For \(N=5\), averages near the middle are more common.

For \(N=30\), the histogram is close to a normal curve.

This is why averages and sums often look approximately normal in physical measurement: many small effects combine.

## 8. CLT caution: small N is not always enough

The chapter says normality can be a good approximation even for averages of 4 or 5 values. That can be true, but it is not guaranteed.

If the original distribution is strongly skewed or heavy-tailed, averages of 4 or 5 can still be skewed.

In [ ]:
# Exponential distribution is skewed
def sample_exponential_means(N, n_samples=50_000):
    samples = rng.exponential(scale=1.0, size=(n_samples, N))
    return samples.mean(axis=1)

for N in [1, 5, 30, 100]:
    means = sample_exponential_means(N)
    plt.figure()
    plt.hist(means, bins=60)
    plt.xlabel("sample mean")
    plt.ylabel("count")
    plt.title(f"Averages of N={N} exponential values")
    plt.show()

For skewed distributions, larger \(N\) may be needed before the normal approximation is good.

This matters in benchmarking: if run times or latencies have rare stalls, averages may converge slowly.

## 9. Weighted average from several methods

Suppose three methods measure the same physical constant.

Each method is unbiased, but each has a different standard deviation and number of replicates.

The method average with smaller variance should get more weight.

If method \(A\)'s sample mean has variance:

\[
V(\bar{X}_A)=\frac{\sigma_A^2}{N_A}
\]

then its inverse-variance weight is:

\[
w_A=\frac{1}{V(\bar{X}_A)}=\frac{N_A}{\sigma_A^2}
\]

The weighted estimate is:

\[
\hat{\mu}=
\frac{w_A\bar{x}_A+w_B\bar{x}_B+w_C\bar{x}_C}
{w_A+w_B+w_C}
\]

In [ ]:
true_constant = 9.81

methods = pd.DataFrame({
    "method": ["A", "B", "C"],
    "sigma": [0.20, 0.50, 0.10],
    "N": [5, 10, 3],
})

# Simulate observations for each method
rows = []
for _, row in methods.iterrows():
    obs = rng.normal(true_constant, row["sigma"], size=int(row["N"]))
    rows.append({
        "method": row["method"],
        "sigma": row["sigma"],
        "N": int(row["N"]),
        "sample_mean": obs.mean(),
        "variance_of_sample_mean": row["sigma"]**2 / row["N"],
        "weight": row["N"] / row["sigma"]**2,
    })

method_results = pd.DataFrame(rows)
method_results

In [ ]:
simple_average = method_results["sample_mean"].mean()

weighted_average = (
    method_results["weight"] * method_results["sample_mean"]
).sum() / method_results["weight"].sum()

simple_average, weighted_average, true_constant

### Interpretation

The weighted average gives more influence to methods with smaller uncertainty.

This is common in physics when multiple experimental methods estimate the same constant.

Caution: inverse-variance weighting assumes the methods are unbiased and their variances are correctly known.

## 10. Pooling variances

Now suppose several chemical samples have different true means but the **same analytical precision**.

Example:

- sample A has true concentration 10
- sample B has true concentration 20
- sample C has true concentration 50
- sample D has true concentration 80

The means differ, but the method's repeatability standard deviation is the same.

Then we can pool within-sample variance estimates.

In [ ]:
true_sigma_method = 0.8

sample_specs = [
    ("A", 10.0, 4),
    ("B", 20.0, 2),
    ("C", 50.0, 2),
    ("D", 80.0, 3),
]

records = []
all_values = []
for name, mu, N in sample_specs:
    values = rng.normal(mu, true_sigma_method, size=N)
    mean = values.mean()
    ss = np.sum((values - mean)**2)
    df = N - 1
    s2 = ss / df
    records.append({
        "sample": name,
        "N": N,
        "mean": mean,
        "sum_of_squares": ss,
        "df": df,
        "variance_estimate": s2
    })
    for v in values:
        all_values.append((name, v))

pool_df = pd.DataFrame(records)
pool_df

In [ ]:
pooled_variance = pool_df["sum_of_squares"].sum() / pool_df["df"].sum()
pooled_sd = math.sqrt(pooled_variance)

weighted_average_of_variances = (
    pool_df["df"] * pool_df["variance_estimate"]
).sum() / pool_df["df"].sum()

pooled_variance, weighted_average_of_variances, pooled_sd

The pooled variance is:

\[
s_p^2=\frac{SS_A+SS_B+SS_C+SS_D}{df_A+df_B+df_C+df_D}
\]

Equivalently:

\[
s_p^2=
\frac{df_A s_A^2+df_B s_B^2+df_C s_C^2+df_D s_D^2}
{df_A+df_B+df_C+df_D}
\]

This is a weighted average of variance estimates, weighted by degrees of freedom.

### Important warning

Pooling is meaningful only if the common-variance assumption is plausible.

If measurement variance grows with concentration, pooling absolute variances may be wrong. In that case, coefficient of variation or a transformed scale may be more appropriate.

## 11. Demonstrating bad pooling

Suppose standard deviation grows with concentration:

\[
\sigma = 0.05\mu
\]

Then high-concentration samples are noisier in absolute terms.

In [ ]:
bad_records = []
for name, mu, N in sample_specs:
    sigma = 0.05 * mu
    values = rng.normal(mu, sigma, size=20)
    mean = values.mean()
    sd = values.std(ddof=1)
    bad_records.append({
        "sample": name,
        "true_mean": mu,
        "sample_mean": mean,
        "sample_sd": sd,
        "CV_percent": 100 * sd / mean
    })

bad_pool_df = pd.DataFrame(bad_records)
bad_pool_df

The standard deviations differ a lot, but the coefficient of variation may be more stable.

This is a common issue in chemistry: absolute measurement error and relative measurement error are different assumptions.

## 12. Derived measurements and bias

Many experimental quantities are not directly measured.

Examples:

- density = mass / volume
- percent composition from titration volumes
- concentration from absorbance
- area from radius

A derived measurement can be biased even if the direct measurement is unbiased, if the transformation is nonlinear.

## 13. Nonlinear bias: area of a circle

True area:

\[
A=\pi R^2
\]

Measured radius:

\[
r = R + e
\]

where:

\[
E(e)=0
\]

Derived area:

\[
a=\pi r^2
\]

Then:

\[
a = \pi(R+e)^2
\]

\[
a = \pi R^2 + 2\pi Re + \pi e^2
\]

Taking expectation:

\[
E(a)=\pi R^2 + 2\pi R E(e) + \pi E(e^2)
\]

If \(E(e)=0\):

\[
E(a)=A+\pi V(e)
\]

So the derived area is biased upward by:

\[
B(a)=\pi V(e)
\]

In [ ]:
R = 1.0
sigma_r = 0.01
n = 1_000_000

e = rng.normal(0, sigma_r, size=n)
r = R + e

A_true = math.pi * R**2
a_measured = math.pi * r**2

observed_bias = a_measured.mean() - A_true
theoretical_bias = math.pi * sigma_r**2

observed_bias, theoretical_bias, 100 * theoretical_bias / A_true

With \(R=1\) m and \(\sigma_r=0.01\) m:

\[
V(e)=0.0001
\]

\[
B(a)=\pi(0.0001)
\]

Relative bias:

\[
\frac{B(a)}{A}=\frac{\pi(0.0001)}{\pi}=0.0001=0.01\%
\]

So random radius error of 1% creates area bias of only 0.01%.

## 14. Random error in the derived area

The bias was small. But the random error in the area is larger.

Use linearization:

\[
A=\pi R^2
\]

\[
\frac{dA}{dR}=2\pi R
\]

So:

\[
\sigma_A \approx 2\pi R\sigma_R
\]

Relative standard deviation:

\[
\frac{\sigma_A}{A}
=
\frac{2\pi R\sigma_R}{\pi R^2}
=
2\frac{\sigma_R}{R}
\]

If radius CV is 1%, area CV is about 2%.

In [ ]:
observed_sd_area = a_measured.std(ddof=1)
approx_sd_area = 2 * math.pi * R * sigma_r

observed_cv_area = observed_sd_area / A_true
approx_cv_area = approx_sd_area / A_true

pd.DataFrame({
    "quantity": ["area sd", "area CV percent"],
    "observed": [observed_sd_area, 100 * observed_cv_area],
    "linearized_approximation": [approx_sd_area, 100 * approx_cv_area]
})

The random uncertainty is about 2% CV, while the nonlinear bias is only 0.01%.

That is why the chapter says the bias is often negligible compared with random error.

## 15. Law of propagation of errors: one variable

For:

\[
Z=f(X)
\]

small errors propagate approximately as:

\[
\delta Z \approx f'(X)\delta X
\]

Therefore:

\[
V(Z)\approx [f'(X)]^2 V(X)
\]

This is also called the **delta method** in modern statistics.

## 16. Law of propagation of errors: several variables

For:

\[
u=f(x,y,z,\ldots)
\]

with independent measurement errors:

\[
V(u)\approx
\left(\frac{\partial f}{\partial x}\right)^2V(x)
+
\left(\frac{\partial f}{\partial y}\right)^2V(y)
+
\left(\frac{\partial f}{\partial z}\right)^2V(z)
+\cdots
\]

If variables are correlated, covariance terms must be added:

\[
2\frac{\partial f}{\partial x}
\frac{\partial f}{\partial y}
\operatorname{Cov}(x,y)
\]

## 17. Chemistry example: iron by back-titration

The chapter gives:

\[
\%Fe = 0.01117(D-F)
\]

where:

- \(D\): ml of potassium dichromate
- \(F\): ml of ferrous ammonium sulfate
- both are titration volumes
- 0.01117 is a chemistry/stoichiometry/sample-size constant

Suppose:

\[
D=21.0,\quad F=0.8
\]

Then:

\[
\%Fe = 0.01117(21.0-0.8)=0.226
\]

If both titration volumes have standard deviation 0.1 ml:

\[
V(D)=V(F)=0.01
\]

Because the formula is linear and errors are independent:

\[
V(\%Fe)=(0.01117)^2V(D)+(0.01117)^2V(F)
\]

In [ ]:
D = 21.0
F = 0.8
c = 0.01117
sd_D = 0.1
sd_F = 0.1

fe = c * (D - F)

var_fe = c**2 * sd_D**2 + c**2 * sd_F**2
sd_fe = math.sqrt(var_fe)
cv_fe_percent = 100 * sd_fe / fe

fe, sd_fe, cv_fe_percent

The standard deviation of the derived percent iron is about:

\[
0.00156
\]

The coefficient of variation is about:

\[
0.7\%
\]

Even though \(F\) is subtracted, its variance contributes positively.

## 18. Specific gravity / density as a quotient

Specific gravity or density-like quantity:

\[
\rho=\frac{P}{W}
\]

where:

- \(P\): weight/mass
- \(W\): volume

For independent errors, the propagation formula gives:

\[
(CV_\rho)^2 \approx (CV_P)^2 + (CV_W)^2
\]

Relative errors add in quadrature for products and quotients.

In [ ]:
P = 100.0       # mass/weight
W = 20.0        # volume
rho = P / W

cv_P = 0.01     # 1%
cv_W = 0.02     # 2%

sd_P = cv_P * P
sd_W = cv_W * W

cv_rho_approx = math.sqrt(cv_P**2 + cv_W**2)
sd_rho_approx = cv_rho_approx * rho

rho, cv_rho_approx, 100 * cv_rho_approx, sd_rho_approx

In [ ]:
# Simulation check
n = 1_000_000
P_obs = rng.normal(P, sd_P, size=n)
W_obs = rng.normal(W, sd_W, size=n)

rho_obs = P_obs / W_obs

pd.DataFrame({
    "quantity": ["rho mean", "rho sd", "rho CV percent"],
    "simulation": [rho_obs.mean(), rho_obs.std(ddof=1), 100 * rho_obs.std(ddof=1) / rho_obs.mean()],
    "linearized_approx": [rho, sd_rho_approx, 100 * cv_rho_approx]
})

The approximation works well when relative errors are small.

For large errors or strongly nonlinear functions, simulation may be safer.

## 19. Multiplicative formulas

For:

\[
u=\frac{xyz}{pq}
\]

the approximate relative variance is:

\[
(CV_u)^2
\approx
(CV_x)^2+(CV_y)^2+(CV_z)^2+(CV_p)^2+(CV_q)^2
\]

Signs and numerator/denominator positions do not matter at first order for relative variance.

In [ ]:
components = pd.DataFrame({
    "quantity": ["x", "y", "z", "p", "q"],
    "CV_percent": [1.0, 2.0, 1.5, 0.5, 3.0]
})

components["CV"] = components["CV_percent"] / 100
combined_cv = math.sqrt(np.sum(components["CV"]**2))

components, 100 * combined_cv

The derived quantity has CV equal to the square root of the sum of squared component CVs.

## 20. Benchmarking example: TPS as a derived quantity

A throughput measurement is:

\[
TPS = \frac{\text{transactions}}{\text{seconds}}
\]

This is a quotient, like density.

If elapsed time uncertainty is negligible, most TPS variability comes from transaction-count variability.

If elapsed time is noisy or the run is very short, both numerator and denominator matter.

In [ ]:
transactions = 1_200_000
seconds = 60.0
tps = transactions / seconds

# Suppose transaction count has 1.5% CV and time has 0.1% CV
cv_tx = 0.015
cv_time = 0.001

cv_tps = math.sqrt(cv_tx**2 + cv_time**2)

tps, 100 * cv_tps

For fixed-duration pgbench runs, elapsed-time variation is often much smaller than run-to-run throughput variation. But for very short benchmarks, startup/shutdown/timing effects can matter.

## 21. Nonlinear ratio caution: average of ratios vs ratio of totals

For derived quantities like TPS, be careful when averaging ratios.

Example:

\[
TPS_i = \frac{T_i}{S_i}
\]

The average of interval TPS values is not always the same as:

\[
\frac{\sum_i T_i}{\sum_i S_i}
\]

In [ ]:
intervals = pd.DataFrame({
    "transactions": [1000, 1000],
    "seconds": [1, 2]
})
intervals["TPS"] = intervals["transactions"] / intervals["seconds"]

average_of_tps = intervals["TPS"].mean()
ratio_of_totals = intervals["transactions"].sum() / intervals["seconds"].sum()

intervals, average_of_tps, ratio_of_totals

Average of interval TPS:

\[
(1000 + 500)/2 = 750
\]

Ratio of totals:

\[
2000/3 \approx 666.7
\]

Both answer different questions. For unequal intervals, ratio of totals is usually the correct aggregate throughput.

## 22. Parameter-estimation viewpoint

The chapter ends by shifting perspective.

Instead of saying:

> measured radius \(r\) gives derived area \(a=\pi r^2\)

we can say:

> the true radius or true area is an unknown parameter, and measurements give evidence about it.

This leads toward least squares.

Example: fitting a line to calibration data.

\[
y = \alpha + \beta x + \epsilon
\]

The unknown constants \(\alpha\) and \(\beta\) are parameters. We estimate them from noisy measurements.

In [ ]:
# Simulate calibration data
x = np.linspace(0, 10, 20)
alpha_true = 2.0
beta_true = 3.0
noise = rng.normal(0, 1.0, size=len(x))
y = alpha_true + beta_true * x + noise

# Least squares estimate
Xmat = np.column_stack([np.ones_like(x), x])
alpha_hat, beta_hat = np.linalg.lstsq(Xmat, y, rcond=None)[0]

plt.figure()
plt.scatter(x, y, label="observed data")
plt.plot(x, alpha_true + beta_true*x, label="true line")
plt.plot(x, alpha_hat + beta_hat*x, linestyle="--", label="least-squares fit")
plt.xlabel("known x")
plt.ylabel("measured y")
plt.title("Parameter estimation by least squares")
plt.legend()
plt.show()

alpha_hat, beta_hat

This is the broader statistical-modeling view:

- observations are noisy
- parameters are unknown
- choose parameter estimates that best explain the observations

Chapter 4 prepares for this by showing how errors propagate through formulas.

## 23. Important cautions about Chapter 4

### 1. Random selection does not always imply independence

If samples are taken without replacement from a finite population, or if measurements share drift/cache/temperature effects, observations are not fully independent.

### 2. The CLT is not magic

Averages often approach normality, but the required sample size depends on skewness, tails, and dependence.

### 3. Five observations may not be enough

The chapter says normal approximation can be good with four or five observations in some cases. That is not universal.

### 4. Error propagation is approximate for nonlinear functions

The derivative-based formula is a first-order Taylor approximation. It works best when errors are small.

### 5. Include covariance terms when errors are correlated

The simple quadrature formula assumes independent errors.

### 6. Pooling variance requires a common-variance assumption

If variances differ meaningfully across groups, pooled variance can be misleading.

## 24. Final checklist

After Chapter 4, you should be able to answer:

1. Is the derived quantity linear or nonlinear?
2. Are the input measurement errors independent?
3. If not, what covariance terms matter?
4. What is the expected value of the derived quantity?
5. What is its variance or standard deviation?
6. Does averaging reduce uncertainty by \(\sqrt{N}\)?
7. Is the central limit theorem applicable?
8. Should estimates be weighted by inverse variance?
9. Is pooling variances justified?
10. Can nonlinear transformation introduce bias?
11. Is first-order error propagation accurate enough?
12. Would simulation or least squares be a better approach?

These questions are directly useful for physical experiments, chemical analysis, and systems benchmarking.